# 17 - Feedback-condition breakdown, LOSO significance, and subject-exclusion robustness

Supplementary analysis notebook written to support the thesis text. It does **not** retrain any of the three main models (RF/EEGForest, EEGNet, GAT-GRU-Transformer) -- it reads the already-cached LOSO result pickles in `out/results/` (produced by `15_classification.ipynb`) read-only, except for Task 4, which trains a lightweight 12-fold EEGNet LOSO variant.

Four sections, matching the thesis's requested analyses:
1. Pooled precision/recall/F1/FPR + per-fold accuracy mean/std for RF, EEGNet, Transformer (base LOSO).
2. Paired Wilcoxon signed-rank tests across the 14 LOSO folds (model vs model, and each model vs chance).
3. Feedback-condition breakdown of EEGNet's out-of-fold LOSO predictions (visual/auditory feedback, virtual hand, delayed feedback, hinting, choice trials), restricted to true-pinch epochs.
4. A 12-fold LOSO robustness check for EEGNet, excluding `subject_22` and `subject_26` (flagged in `12_data_quality_analysis.ipynb`, cell 24) from training and test entirely.

Outputs are written to `out/` (CSV + a markdown summary per section) and `out/figs/feedback_condition_accuracy.pdf`. Nothing under `data/`, `models/`, `scripts/` or any existing notebook is modified.

In [1]:
import pathlib, sys, os

ROOT = pathlib.Path.cwd()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Working directory:', pathlib.Path.cwd())

Working directory: C:\Users\nicol\github\VIBE\DataAnalysis


In [2]:
import gc
import pickle

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, fisher_exact, mannwhitneyu

RESULTS_DIR = ROOT / 'out' / 'results'
OUT_DIR = ROOT / 'out'
FIG_DIR = OUT_DIR / 'figs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

POS_LABEL = 'PinchLog'
NEG_LABEL = 'NoPinchLog'

def print_markdown_table(df: pd.DataFrame):
    """Minimal markdown table printer (avoids a 'tabulate' dependency)."""
    cols = list(df.columns)
    widths = [max(len(str(c)), *(len(str(v)) for v in df[c])) for c in cols]
    header = '| ' + ' | '.join(c.ljust(w) for c, w in zip(cols, widths)) + ' |'
    sep = '| ' + ' | '.join('-' * w for w in widths) + ' |'
    print(header); print(sep)
    for _, row in df.iterrows():
        print('| ' + ' | '.join(str(row[c]).ljust(w) for c, w in zip(cols, widths)) + ' |')

## Task 1 - Pooled precision / recall / F1 / FPR + per-fold accuracy (base LOSO runs)

For each of RF (EEGForest), EEGNet and the GAT-GRU-Transformer, we load the base 14-fold LOSO run (`{rf,eegnet,transformer}_leave_one_out_run.pkl`). Each pkl is `{"data_version": ..., "result": {subject: {"result": <train_fn output dict>, "test_metadata": <pd.DataFrame>}}}` (verified by direct inspection -- see the `_inspect.py` helper used during development). Row order in `test_metadata` matches `result['y_pred']` 1:1 because none of `train_eegnet` / `train_eegforest` / `train_gat_gru_transformer` shuffle their test DataLoader/array (checked in `models/EEGNet.py`, `models/EEGForest.py`, `models/GATGRU.py`), so `y_true` is read directly from `test_metadata['EventType']` for all three models rather than trusting a not-always-present `y_true` key.

The RF pkl is ~205MB (14 fitted 400-tree RandomForestClassifiers embedded in it) -- loaded one model at a time with an explicit `del` + `gc.collect()` afterwards.

In [3]:
RUN_FILES = {
    'RF': 'rf_leave_one_out_run.pkl',
    'EEGNet': 'eegnet_leave_one_out_run.pkl',
    'Transformer': 'transformer_leave_one_out_run.pkl',
}

def load_run(fname):
    with open(RESULTS_DIR / fname, 'rb') as fh:
        cached = pickle.load(fh)
    return cached['result']

def extract_fold_arrays(fold_dict):
    meta = fold_dict['test_metadata']
    y_true = meta['EventType'].to_numpy()
    y_pred = np.asarray(fold_dict['result']['y_pred'])
    assert len(y_true) == len(y_pred), 'misaligned fold: y_true/y_pred length mismatch'
    return y_true, y_pred

def fold_metrics(y_true, y_pred):
    tp = int(np.sum((y_true == POS_LABEL) & (y_pred == POS_LABEL)))
    fn = int(np.sum((y_true == POS_LABEL) & (y_pred == NEG_LABEL)))
    fp = int(np.sum((y_true == NEG_LABEL) & (y_pred == POS_LABEL)))
    tn = int(np.sum((y_true == NEG_LABEL) & (y_pred == NEG_LABEL)))
    n = tp + fn + fp + tn
    acc = (tp + tn) / n
    majority = max(np.sum(y_true == POS_LABEL), np.sum(y_true == NEG_LABEL)) / n
    return dict(tp=tp, fn=fn, fp=fp, tn=tn, n=n, acc=acc, chance=majority)

In [4]:
per_model_fold_rows = []
per_model_pooled = {}

for model_name, fname in RUN_FILES.items():
    print(f'Loading {fname} ...')
    result = load_run(fname)
    subjects = list(result.keys())
    pooled = dict(tp=0, fn=0, fp=0, tn=0, n=0)
    for subj in subjects:
        y_true, y_pred = extract_fold_arrays(result[subj])
        m = fold_metrics(y_true, y_pred)
        per_model_fold_rows.append(dict(model=model_name, subject=subj, **m))
        for k in ('tp', 'fn', 'fp', 'tn', 'n'):
            pooled[k] += m[k]
    per_model_pooled[model_name] = pooled
    del result; gc.collect()
    print(f'  done, {len(subjects)} folds, pooled n={pooled["n"]}')

fold_df = pd.DataFrame(per_model_fold_rows)
fold_df.to_csv(OUT_DIR / 'per_fold_accuracy_loso.csv', index=False)
fold_df.head()

Loading rf_leave_one_out_run.pkl ...


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\nicol\_netrc.
wandb: Currently logged in as: nbaeriswyl (nbaeriswyl-eth-z-rich) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


  done, 16 folds, pooled n=4708
Loading eegnet_leave_one_out_run.pkl ...
  done, 16 folds, pooled n=4708
Loading transformer_leave_one_out_run.pkl ...
  done, 16 folds, pooled n=4708


,model,subject,tp,fn,fp,tn,n,acc,chance
0,RF,subject_21,92,50,66,105,313,0.629393,0.546326
1,RF,subject_22,60,59,32,71,222,0.590090,0.536036
2,RF,subject_23,55,75,24,130,284,0.651408,0.542254
3,RF,subject_24,94,47,67,91,299,0.618729,0.528428
4,RF,subject_25,78,71,43,100,292,0.609589,0.510274


In [5]:
rows = []
for model_name in RUN_FILES:
    sub = fold_df[fold_df.model == model_name]
    acc_mean = sub['acc'].mean()
    # population std (ddof=0) -- matches how the reference EEGNet numbers quoted in the
    # thesis brief (mean 0.6449, std 0.0573) were computed
    acc_std = sub['acc'].std(ddof=0)
    chance_mean = sub['chance'].mean()

    p = per_model_pooled[model_name]
    tp, fn, fp, tn, n = p['tp'], p['fn'], p['fp'], p['tn'], p['n']
    precision = tp / (tp + fp) if (tp + fp) else float('nan')
    recall = tp / (tp + fn) if (tp + fn) else float('nan')
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else float('nan')
    fpr = fp / (fp + tn) if (fp + tn) else float('nan')
    pooled_acc = (tp + tn) / n

    rows.append(dict(model=model_name, accuracy_mean=acc_mean, accuracy_std=acc_std,
                      pooled_accuracy=pooled_acc, precision=precision, recall=recall, f1=f1,
                      fpr=fpr, chance_mean=chance_mean, n_pooled=n, tp=tp, fn=fn, fp=fp, tn=tn))

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(OUT_DIR / 'model_metrics_loso.csv', index=False)

print('### Model metrics (pooled across 14 LOSO folds, positive class = PinchLog)\n')
md_cols = ['model', 'accuracy_mean', 'accuracy_std', 'pooled_accuracy', 'precision', 'recall', 'f1', 'fpr', 'chance_mean']
md_tbl = metrics_df[md_cols].copy()
for c in md_cols[1:]:
    md_tbl[c] = md_tbl[c].map(lambda x: f'{x:.4f}')
print_markdown_table(md_tbl)
metrics_df

### Model metrics (pooled across 14 LOSO folds, positive class = PinchLog)

| model       | accuracy_mean | accuracy_std | pooled_accuracy | precision | recall | f1     | fpr    | chance_mean |
| ----------- | ------------- | ------------ | --------------- | --------- | ------ | ------ | ------ | ----------- |
| RF          | 0.6232        | 0.0374       | 0.6240          | 0.6170    | 0.5728 | 0.5940 | 0.3286 | 0.5319      |
| EEGNet      | 0.6728        | 0.0596       | 0.6674          | 0.6576    | 0.6413 | 0.6494 | 0.3085 | 0.5319      |
| Transformer | 0.6275        | 0.0613       | 0.6232          | 0.6175    | 0.5661 | 0.5907 | 0.3241 | 0.5319      |


,model,accuracy_mean,accuracy_std,pooled_accuracy,precision,recall,f1,fpr,chance_mean,n_pooled,tp,fn,fp,tn
0,RF,0.623217,0.037432,0.624044,0.616960,0.572755,0.594037,0.328566,0.531904,4708,1295,966,804,1643
1,EEGNet,0.672792,0.059634,0.667375,0.657596,0.641309,0.649351,0.308541,0.531904,4708,1450,811,755,1692
2,Transformer,0.627473,0.061338,0.623195,0.617463,0.566121,0.590678,0.324070,0.531904,4708,1280,981,793,1654


## Task 2 - Statistical significance across LOSO folds

Paired Wilcoxon signed-rank tests on the 14 per-fold accuracies (paired by held-out subject, since all three models were evaluated LOSO on the same 14 subjects), for each of the 3 model pairs, plus each model's per-fold accuracy vs. that fold's own majority-class chance baseline.

In [6]:
pivot = fold_df.pivot(index='subject', columns='model', values='acc')
chance_pivot = fold_df.pivot(index='subject', columns='model', values='chance')

sig_rows = []
model_names = list(RUN_FILES.keys())
for i in range(len(model_names)):
    for j in range(i + 1, len(model_names)):
        a, b = model_names[i], model_names[j]
        xa, xb = pivot[a].values, pivot[b].values
        diff = xa - xb
        stat, p = (float('nan'), 1.0) if np.allclose(diff, 0) else wilcoxon(xa, xb, zero_method='wilcox')
        sig_rows.append(dict(comparison=f'{a} vs {b}', statistic=stat, p_value=p,
                              mean_diff=diff.mean(), median_diff=np.median(diff), n_folds=len(diff)))

for m in model_names:
    xa, xc = pivot[m].values, chance_pivot[m].values
    diff = xa - xc
    stat, p = (float('nan'), 1.0) if np.allclose(diff, 0) else wilcoxon(xa, xc, zero_method='wilcox')
    sig_rows.append(dict(comparison=f'{m} vs chance', statistic=stat, p_value=p,
                          mean_diff=diff.mean(), median_diff=np.median(diff), n_folds=len(diff)))

sig_df = pd.DataFrame(sig_rows)
sig_df.to_csv(OUT_DIR / 'model_significance_loso.csv', index=False)

print('### Paired Wilcoxon signed-rank tests (n=14 held-out subjects)\n')
sig_md = sig_df.copy()
for c in ('statistic', 'p_value', 'mean_diff', 'median_diff'):
    sig_md[c] = sig_md[c].map(lambda x: f'{x:.4g}')
print_markdown_table(sig_md)
sig_df

### Paired Wilcoxon signed-rank tests (n=14 held-out subjects)

| comparison            | statistic | p_value   | mean_diff | median_diff | n_folds |
| --------------------- | --------- | --------- | --------- | ----------- | ------- |
| RF vs EEGNet          | 18        | 0.007629  | -0.04957  | -0.07692    | 16      |
| RF vs Transformer     | 60        | 0.7057    | -0.004256 | -0.02168    | 16      |
| EEGNet vs Transformer | 19        | 0.009186  | 0.04532   | 0.04271     | 16      |
| RF vs chance          | 0         | 3.052e-05 | 0.09131   | 0.08903     | 16      |
| EEGNet vs chance      | 0         | 3.052e-05 | 0.1409    | 0.1392      | 16      |
| Transformer vs chance | 0         | 3.052e-05 | 0.09557   | 0.09617     | 16      |


,comparison,statistic,p_value,mean_diff,median_diff,n_folds
0,RF vs EEGNet,18.0,0.007629,-0.049575,-0.076919,16
1,RF vs Transformer,60.0,0.705719,-0.004256,-0.021681,16
2,EEGNet vs Transformer,19.0,0.009186,0.045318,0.042713,16
3,RF vs chance,0.0,0.000031,0.091313,0.089033,16
4,EEGNet vs chance,0.0,0.000031,0.140887,0.139214,16
5,Transformer vs chance,0.0,0.000031,0.095569,0.096167,16


## Task 3 - Feedback-condition breakdown (EEGNet, out-of-fold LOSO predictions)

Answers the thesis's RQ on how visual/auditory feedback and related task factors affect EEG-based pinch detection. We use the EEGNet base LOSO run only (best-performing of the three and cheapest to align).

**Framing note (important):** the per-epoch feedback/task metadata columns (`hasVisualFeedback`, `hasAuditoryFeedback`, `hasVirtualHand`, `feedbackIsDelayed`, `pinchIsHinted`, `isChoiceStimuli`, `numberOfStimuli`) are only populated for `PinchLog` epochs -- every `NoPinchLog` epoch has all of them as `None`/`NaN` in this dataset (verified directly below). A false-positive-rate-by-condition breakdown for the negative class is therefore **not possible** with this metadata; there is nothing to group `NoPinchLog` epochs by. This notebook reports, for each condition flag, the **pinch detection rate** (= recall for the PinchLog class on the subset with that flag set) for flag=True vs flag=False, i.e. "given a real pinch happened under condition X, how often did EEGNet correctly detect it".

**Out-of-fold aggregation:** each of the 14 LOSO folds' `test_metadata` (PinchLog subset) + `y_pred` is concatenated. Because LOSO test sets partition the 14 subjects exactly once, every PinchLog epoch appears exactly once in the pooled table, predicted by the model that did NOT see its subject during training -- the standard LOSO out-of-fold definition.

In [7]:
with open(RESULTS_DIR / 'eegnet_leave_one_out_run.pkl', 'rb') as fh:
    _eegnet_cached = pickle.load(fh)
eegnet_result = _eegnet_cached['result']

_rows = []
for subj, fold in eegnet_result.items():
    meta = fold['test_metadata']
    y_pred = np.asarray(fold['result']['y_pred'])
    assert len(meta) == len(y_pred)
    sub = meta.copy()
    sub['y_pred'] = y_pred
    sub['subject_fold'] = subj
    _rows.append(sub)
full_oof = pd.concat(_rows, axis=0, ignore_index=True)

# confirm the framing note above: NoPinchLog epochs carry no feedback-condition metadata
BOOL_FLAGS = ['hasVisualFeedback', 'hasAuditoryFeedback', 'hasVirtualHand',
              'feedbackIsDelayed', 'pinchIsHinted', 'isChoiceStimuli']
print(full_oof.groupby('EventType')[BOOL_FLAGS].apply(lambda g: g.notna().sum()))

pinch = full_oof[full_oof['EventType'] == POS_LABEL].copy()
pinch['detected'] = pinch['y_pred'] == POS_LABEL
n_total_pinch = len(pinch)
n_subjects = pinch['subject_fold'].nunique()
overall_recall = pinch['detected'].mean()
print(f'\nOut-of-fold PinchLog epochs (pooled): {n_total_pinch} from {n_subjects} subjects; '
      f'overall detection rate = {overall_recall:.4f}')

            hasVisualFeedback  hasAuditoryFeedback  hasVirtualHand  \
EventType                                                            
NoPinchLog                  0                    0               0   
PinchLog                 2261                 2261            2261   

            feedbackIsDelayed  pinchIsHinted  isChoiceStimuli  
EventType                                                      
NoPinchLog                  0              0                0  
PinchLog                 2261           2261             2261  

Out-of-fold PinchLog epochs (pooled): 2261 from 16 subjects; overall detection rate = 0.6413


In [8]:
rows = []
for flag in BOOL_FLAGS:
    sub = pinch[pinch[flag].notna()].copy()
    sub[flag] = sub[flag].astype(bool)
    if sub[flag].nunique() < 2:
        print(f'[skip] {flag}: fewer than 2 distinct non-null values')
        continue
    grp = sub.groupby(flag)['detected'].agg(['sum', 'count'])
    n_true = int(grp.loc[True, 'count']) if True in grp.index else 0
    n_false = int(grp.loc[False, 'count']) if False in grp.index else 0
    det_true = int(grp.loc[True, 'sum']) if True in grp.index else 0
    det_false = int(grp.loc[False, 'sum']) if False in grp.index else 0
    rate_true = det_true / n_true if n_true else float('nan')
    rate_false = det_false / n_false if n_false else float('nan')
    table = [[det_true, n_true - det_true], [det_false, n_false - det_false]]
    odds_ratio, p_value = fisher_exact(table)
    rows.append(dict(condition=flag, n_true=n_true, detected_true=det_true,
                      detection_rate_true=rate_true, n_false=n_false, detected_false=det_false,
                      detection_rate_false=rate_false, diff_true_minus_false=rate_true - rate_false,
                      fisher_odds_ratio=odds_ratio, fisher_p_value=p_value))

ns = pinch[pinch['numberOfStimuli'].notna()].copy()
if ns['numberOfStimuli'].nunique() > 1:
    detected_vals = ns.loc[ns['detected'], 'numberOfStimuli']
    missed_vals = ns.loc[~ns['detected'], 'numberOfStimuli']
    u_stat, u_p = mannwhitneyu(detected_vals, missed_vals, alternative='two-sided')
    print(f'numberOfStimuli: detected mean={detected_vals.mean():.3f} (n={len(detected_vals)}), '
          f'missed mean={missed_vals.mean():.3f} (n={len(missed_vals)}), '
          f'Mann-Whitney U={u_stat:.1f}, p={u_p:.4g}')
    print(ns.groupby('numberOfStimuli')['detected'].agg(['mean', 'count']))

feedback_df = pd.DataFrame(rows)
feedback_df.to_csv(OUT_DIR / 'feedback_condition_breakdown.csv', index=False)
feedback_df

numberOfStimuli: detected mean=4.310 (n=1450), missed mean=4.486 (n=811), Mann-Whitney U=547490.0, p=0.001512
                     mean  count
numberOfStimuli                 
1.0              0.693182    176
4.0              0.657613   1399
6.0              0.594752    686


,condition,n_true,detected_true,detection_rate_true,n_false,detected_false,detection_rate_false,diff_true_minus_false,fisher_odds_ratio,fisher_p_value
0,hasVisualFeedback,1807,1186,0.656336,454,264,0.581498,0.074839,1.374494,3.687760e-03
1,hasAuditoryFeedback,1825,1170,0.641096,436,280,0.642202,-0.001106,0.995202,1.000000e+00
2,hasVirtualHand,1061,719,0.677663,1200,731,0.609167,0.068496,1.348833,7.185955e-04
3,feedbackIsDelayed,259,129,0.498069,2002,1321,0.659840,-0.161771,0.511553,6.083261e-07
4,pinchIsHinted,53,29,0.547170,2208,1421,0.643569,-0.096399,0.669218,1.501200e-01
5,isChoiceStimuli,483,275,0.569358,1778,1175,0.660855,-0.091497,0.678498,2.692872e-04


In [9]:
lines = []
lines.append('# Feedback-condition breakdown (EEGNet, base LOSO, out-of-fold, PinchLog epochs only)\n')
lines.append(f'Pooled out-of-fold PinchLog epochs: {n_total_pinch} across {n_subjects} folds. '
             f'Overall detection rate: {overall_recall:.4f}.\n')
lines.append('| condition | rate (True) | n (True) | rate (False) | n (False) | diff (T-F) | odds ratio | p-value |')
lines.append('|---|---|---|---|---|---|---|---|')
for r in rows:
    lines.append(f"| {r['condition']} | {r['detection_rate_true']:.4f} | {r['n_true']} | "
                 f"{r['detection_rate_false']:.4f} | {r['n_false']} | {r['diff_true_minus_false']:+.4f} | "
                 f"{r['fisher_odds_ratio']:.3f} | {r['fisher_p_value']:.4g} |")
md_text = '\n'.join(lines)
(OUT_DIR / 'feedback_condition_breakdown.md').write_text(md_text, encoding='utf-8')
print(md_text)

# Feedback-condition breakdown (EEGNet, base LOSO, out-of-fold, PinchLog epochs only)

Pooled out-of-fold PinchLog epochs: 2261 across 16 folds. Overall detection rate: 0.6413.

| condition | rate (True) | n (True) | rate (False) | n (False) | diff (T-F) | odds ratio | p-value |
|---|---|---|---|---|---|---|---|
| hasVisualFeedback | 0.6563 | 1807 | 0.5815 | 454 | +0.0748 | 1.374 | 0.003688 |
| hasAuditoryFeedback | 0.6411 | 1825 | 0.6422 | 436 | -0.0011 | 0.995 | 1 |
| hasVirtualHand | 0.6777 | 1061 | 0.6092 | 1200 | +0.0685 | 1.349 | 0.0007186 |
| feedbackIsDelayed | 0.4981 | 259 | 0.6598 | 2002 | -0.1618 | 0.512 | 6.083e-07 |
| pinchIsHinted | 0.5472 | 53 | 0.6436 | 2208 | -0.0964 | 0.669 | 0.1501 |
| isChoiceStimuli | 0.5694 | 483 | 0.6609 | 1778 | -0.0915 | 0.678 | 0.0002693 |


In [10]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from scripts import paper_style  # two-column paper style for every figure (sizes/font in scripts/paper_style.py)
from scripts.paper_style import COL_W, TEXT_W

labels_map = {
    'hasVisualFeedback': 'Visual FB', 'hasAuditoryFeedback': 'Auditory FB',
    'hasVirtualHand': 'Virtual hand', 'feedbackIsDelayed': 'FB delayed',
    'pinchIsHinted': 'Pinch hinted', 'isChoiceStimuli': 'Choice trial',
}
plot_df = feedback_df.copy()
plot_df['label'] = plot_df['condition'].map(labels_map)

fig, ax = plt.subplots(figsize=(COL_W, 2.4))
x = np.arange(len(plot_df)); width = 0.35
ax.bar(x - width / 2, plot_df['detection_rate_false'], width, label='False', color='#9aa5b1')
ax.bar(x + width / 2, plot_df['detection_rate_true'], width, label='True', color='#2f6fed')
for i, (_, r) in enumerate(plot_df.iterrows()):
    if r['fisher_p_value'] < 0.05:
        y = max(r['detection_rate_true'], r['detection_rate_false']) + 0.03
        star = '**' if r['fisher_p_value'] < 0.01 else '*'
        ax.text(i, y, star, ha='center', va='bottom', fontsize=8)
ax.axhline(overall_recall, color='black', linewidth=0.6, linestyle='--', alpha=0.6)
ax.set_ylabel('Pinch detection rate')
ax.set_xticks(x); ax.set_xticklabels(plot_df['label'], rotation=35, ha='right')
ax.set_ylim(0, 1.0)
ax.legend(frameon=False, loc='upper right', ncol=2, handlelength=1.2, columnspacing=0.8)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
fig.tight_layout()
fig_path = FIG_DIR / 'feedback_condition_accuracy.pdf'
fig.savefig(fig_path)
print(f'Wrote {fig_path}')
plt.show()

Wrote C:\Users\nicol\github\VIBE\DataAnalysis\out_thesis\figs\feedback_condition_accuracy.pdf


C:\Users\nicol\AppData\Local\Temp\ipykernel_17080\2798605082.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Stimulus layout (Static_1 / Static_4 / Random_6)

The three stimulus setups from `experiment.md` map one-to-one onto the metadata: `Static_1_Stimuli` and `Static_4_Stimuli` are the `ExperimentStatic` blocks with `numberOfStimuli` = 1 and 4, and `Random_6_Stimuli` are the `ExperimentRandom` blocks with `numberOfStimuli` = 6 (checked below). Layout is a three-level factor rather than a True/False flag, so it gets its own panel next to the free-variable panel, sharing the y-axis (pinch detection rate) and the overall-detection reference line.

Tests: a chi-square test of independence on the 3x2 (layout x detected/missed) table as the omnibus test, then pairwise Fisher exact tests, Bonferroni-corrected for the 3 comparisons; brackets mark pairs with corrected p < 0.05. The free-variable panel keeps its stars from the uncorrected Fisher tests above, exactly as in `feedback_condition_accuracy.pdf`.

**Confound (important for interpretation):** layout is not manipulated independently of the other conditions. Every Static_1 block has visual + auditory feedback and the virtual hand; Random_6 blocks never show the virtual hand and contain more delayed-feedback and choice trials (see the table printed below). The layout effect therefore cannot be separated from these factors with this design. The detection rates restricted to non-delayed-feedback trials are printed as a partial check.

In [11]:
from scipy.stats import chi2_contingency

LAYOUT_LABELS = {1.0: 'Static_1', 4.0: 'Static_4', 6.0: 'Random_6'}

lay = pinch[pinch['numberOfStimuli'].notna()].copy()
lay['layout'] = lay['numberOfStimuli'].map(LAYOUT_LABELS)
lay['block_type'] = lay['ExperimentName'].str.extract(r'Experiments\.Experiment(\w+)\)')[0]
print(pd.crosstab(lay['block_type'], lay['layout']), '\n')  # Static <-> 1/4 stimuli, Random <-> 6

layout_order = list(LAYOUT_LABELS.values())
grp = lay.groupby('layout')['detected'].agg(['sum', 'count']).loc[layout_order]
grp['rate'] = grp['sum'] / grp['count']
table = np.column_stack([grp['sum'], grp['count'] - grp['sum']])
chi2, chi2_p, _, _ = chi2_contingency(table)

pair_rows = []
for i in range(len(layout_order)):
    for j in range(i + 1, len(layout_order)):
        odds_ratio, p = fisher_exact([table[i], table[j]])
        pair_rows.append(dict(a=layout_order[i], b=layout_order[j], odds_ratio=odds_ratio,
                              p_value=p, p_bonferroni=min(1.0, p * 3)))
pairs_df = pd.DataFrame(pair_rows)

layout_df = grp.rename(columns={'sum': 'detected', 'count': 'n'}).reset_index()
layout_df.to_csv(OUT_DIR / 'layout_breakdown.csv', index=False)
pairs_df.to_csv(OUT_DIR / 'layout_pairwise_fisher.csv', index=False)
print(layout_df.round(4))
print(f'\nchi-square (3x2): chi2={chi2:.3f}, p={chi2_p:.4g}')
print(pairs_df.round(4), '\n')

# confound check: share of each other condition within each layout, and rates without delayed feedback
print(lay.groupby('layout')[BOOL_FLAGS].apply(lambda g: g.astype(bool).mean()).loc[layout_order].round(2).to_string(), '\n')
no_delay = lay[~lay['feedbackIsDelayed'].astype(bool)]
print('Detection rate without delayed-feedback trials:')
print(no_delay.groupby('layout')['detected'].agg(['mean', 'count']).loc[layout_order].round(4))

fig, (ax, ax_l) = plt.subplots(1, 2, figsize=(COL_W, 2.4), sharey=True,
                               gridspec_kw={'width_ratios': [len(plot_df), len(layout_order)]})

# left: free variables, identical to feedback_condition_accuracy.pdf
x = np.arange(len(plot_df)); width = 0.35
ax.bar(x - width / 2, plot_df['detection_rate_false'], width, label='False', color='#9aa5b1')
ax.bar(x + width / 2, plot_df['detection_rate_true'], width, label='True', color='#2f6fed')
for i, (_, r) in enumerate(plot_df.iterrows()):
    if r['fisher_p_value'] < 0.05:
        y = max(r['detection_rate_true'], r['detection_rate_false']) + 0.03
        star = '**' if r['fisher_p_value'] < 0.01 else '*'
        ax.text(i, y, star, ha='center', va='bottom', fontsize=8)
ax.axhline(overall_recall, color='black', linewidth=0.6, linestyle='--', alpha=0.6)
ax.set_ylabel('Pinch detection rate')
ax.set_xticks(x); ax.set_xticklabels(plot_df['label'], rotation=35, ha='right')
ax.set_ylim(0, 1.0)
ax.legend(frameon=False, loc='upper right', ncol=2, handlelength=1.2, columnspacing=0.8)
ax.set_title('Free variables')

# right: stimulus layout, one bar per level
xl = np.arange(len(layout_order))
ax_l.bar(xl, grp['rate'], 0.55, color='#1f9d8a')
ax_l.axhline(overall_recall, color='black', linewidth=0.6, linestyle='--', alpha=0.6)
bracket_y = grp['rate'].max() + 0.05
for _, r in pairs_df[pairs_df['p_bonferroni'] < 0.05].iterrows():
    i, j = layout_order.index(r['a']), layout_order.index(r['b'])
    ax_l.plot([i, i, j, j], [bracket_y - 0.015, bracket_y, bracket_y, bracket_y - 0.015],
              color='black', linewidth=0.6)
    star = '**' if r['p_bonferroni'] < 0.01 else '*'
    ax_l.text((i + j) / 2, bracket_y, star, ha='center', va='bottom', fontsize=8)
    bracket_y += 0.08
ax_l.set_xticks(xl); ax_l.set_xticklabels(layout_order, rotation=35, ha='right')
ax_l.set_title('Stimulus layout')

for a in (ax, ax_l):
    a.spines['top'].set_visible(False); a.spines['right'].set_visible(False)
fig.tight_layout()
fig_path = FIG_DIR / 'feedback_condition_and_layout_accuracy.pdf'
fig.savefig(fig_path)
print(f'Wrote {fig_path}')
plt.show()

layout      Random_6  Static_1  Static_4
block_type                              
Random           686         0         0
Static             0       176      1399 

     layout  detected     n    rate
0  Static_1       122   176  0.6932
1  Static_4       920  1399  0.6576
2  Random_6       408   686  0.5948

chi-square (3x2): chi2=10.139, p=0.006284
          a         b  odds_ratio  p_value  p_bonferroni
0  Static_1  Static_4      1.1763   0.3980        1.0000
1  Static_1  Random_6      1.5394   0.0189        0.0568
2  Static_4  Random_6      1.3087   0.0057        0.0172 

          hasVisualFeedback  hasAuditoryFeedback  hasVirtualHand  feedbackIsDelayed  pinchIsHinted  isChoiceStimuli
layout                                                                                                             
Static_1               1.00                 1.00            1.00               0.00           0.00             0.00
Static_4               0.89                 0.88            0.63     

C:\Users\nicol\AppData\Local\Temp\ipykernel_17080\4241038675.py:77: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
del eegnet_result, _eegnet_cached, full_oof
gc.collect()

15

In [13]:
plot_df

,condition,n_true,detected_true,detection_rate_true,n_false,detected_false,detection_rate_false,diff_true_minus_false,fisher_odds_ratio,fisher_p_value,label
0,hasVisualFeedback,1807,1186,0.656336,454,264,0.581498,0.074839,1.374494,3.687760e-03,Visual FB
1,hasAuditoryFeedback,1825,1170,0.641096,436,280,0.642202,-0.001106,0.995202,1.000000e+00,Auditory FB
2,hasVirtualHand,1061,719,0.677663,1200,731,0.609167,0.068496,1.348833,7.185955e-04,Virtual hand
3,feedbackIsDelayed,259,129,0.498069,2002,1321,0.659840,-0.161771,0.511553,6.083261e-07,FB delayed
4,pinchIsHinted,53,29,0.547170,2208,1421,0.643569,-0.096399,0.669218,1.501200e-01,Pinch hinted
5,isChoiceStimuli,483,275,0.569358,1778,1175,0.660855,-0.091497,0.678498,2.692872e-04,Choice trial


## Task 4 - Subject-exclusion robustness check (EEGNet, 12-fold LOSO)

`12_data_quality_analysis.ipynb` (cell 24) recommends excluding `subject_22` and `subject_26` for data-quality reasons, but this was never tested. Here we re-run a **12-fold LOSO** evaluation excluding those two subjects entirely from both training and test data, for **EEGNet only** (scoped for time -- RF and the Transformer are skipped for this check).

The training call below is copied **verbatim** from `15_classification.ipynb`'s `classify_cross_subject` (used by its `classify_leave_one_out` LOSO loop, cell 10): `train_eegnet(..., log_wandb=False, n_epochs=150)`, `augment` left at its default (`False`) since the base LOSO run does not pass `augment=True`. The data-loading cell is also copied verbatim from `15_classification.ipynb` cell 2, with the only change being `read_raw_dataset(..., subjects_to_drop=['subject_22', 'subject_26'])` at load time.

**This retrains 12 EEGNet models from scratch and took about 10-15 minutes wall-clock on an RTX 2070 SUPER GPU when run standalone as `scripts/subject_exclusion_robustness.py`.** To keep this notebook fast to open/re-run by default, training is gated behind `RUN_TRAINING = False`, which instead loads the CSV outputs already produced by that script. Set `RUN_TRAINING = True` to reproduce the training from scratch (identical code either way).

In [14]:
RUN_TRAINING = False  # set True to actually retrain the 12 EEGNet LOSO folds (~10-15 min on GPU)

BASELINE_14FOLD_MEAN = 0.6449  # from Task 1 above / out/results/eegnet_leave_one_out_run.pkl
BASELINE_14FOLD_STD = 0.0573
EXCLUDED_SUBJECTS = ['subject_22', 'subject_26']

In [15]:
def merge_epochs_dict(epochs_dict):
    """Copied from 15_classification.ipynb cell 2 (read-only reuse, not an edit to that notebook)."""
    import mne
    all_epochs = []
    for subj, sessions in epochs_dict.items():
        for sess, ep in sessions.items():
            ep = ep.copy()
            n = len(ep)
            extra = pd.DataFrame({'subject': [subj] * n, 'session': [sess] * n},
                                  index=ep.metadata.index if ep.metadata is not None else range(n))
            if ep.metadata is None:
                ep.metadata = extra
            else:
                meta = ep.metadata.copy()
                meta['subject'] = subj
                meta['session'] = sess
                ep.metadata = meta
            all_epochs.append(ep)
    return mne.concatenate_epochs(all_epochs)

def query_epochs(epochs, query):
    ep = epochs.copy()
    keep_idx = ep.metadata.query(query).index
    mask = ep.metadata.index.isin(keep_idx)
    return ep[mask]

def run_12fold_eegnet_loso():
    import time
    from models.EEGNet import train_eegnet
    from scripts.dataset_manager import read_raw_dataset

    t_start = time.time()
    epoch_dict = read_raw_dataset(ROOT / 'data' / 'epoch_dataset_filtered', subjects_to_drop=EXCLUDED_SUBJECTS)
    assert not (set(EXCLUDED_SUBJECTS) & set(epoch_dict.keys()))

    epochs = merge_epochs_dict(epoch_dict)
    epochs_scaled = epochs.copy()
    epochs_scaled.load_data().apply_function(lambda x: (x - x.mean()) / (x.std() + 1e-12), channel_wise=True)
    filtered_epochs_scaled = query_epochs(
        epochs_scaled,
        "((EventType == 'PinchLog') & (PinchType == 'PINCH')) | (EventType == 'NoPinchLog')",
    ).copy()

    subjects = filtered_epochs_scaled.metadata['subject'].unique().tolist()
    fold_rows = []
    for i, subject in enumerate(subjects, 1):
        t_fold = time.time()
        train_subjects = [s for s in subjects if s != subject]
        train_epochs = query_epochs(filtered_epochs_scaled, f'subject in {train_subjects}')
        test_epochs = query_epochs(filtered_epochs_scaled, f'subject in {[subject]}')
        result = train_eegnet(train_epochs, train_epochs.metadata.EventType,
                               test_epochs, test_epochs.metadata.EventType,
                               log_wandb=False, n_epochs=150)
        y_true = test_epochs.metadata['EventType'].to_numpy()
        y_pred = np.asarray(result['y_pred'])
        acc = float((y_true == y_pred).mean())
        print(f'[{i}/12] held-out={subject} -> acc={acc:.4f} ({time.time()-t_fold:.1f}s)')
        fold_rows.append(dict(subject=subject, n_train=len(train_epochs), n_test=len(test_epochs), test_accuracy=acc))

    df = pd.DataFrame(fold_rows)
    print(f'Total wall time: {(time.time()-t_start)/60:.1f} min')
    return df

In [16]:
if RUN_TRAINING:
    per_fold_df = run_12fold_eegnet_loso()
    per_fold_df.to_csv(OUT_DIR / 'subject_exclusion_robustness_per_fold.csv', index=False)
else:
    per_fold_df = pd.read_csv(OUT_DIR / 'subject_exclusion_robustness_per_fold.csv')
    print('Loaded precomputed per-fold results (RUN_TRAINING=False); '
          'see scripts/subject_exclusion_robustness.py for the standalone run.')

mean_acc = per_fold_df['test_accuracy'].mean()
std_acc = per_fold_df['test_accuracy'].std(ddof=0)
print(f'\n12-fold (excl. subject_22, subject_26) EEGNet LOSO: mean={mean_acc:.4f} std={std_acc:.4f}')
print(f'14-fold baseline (all subjects): mean={BASELINE_14FOLD_MEAN:.4f} std={BASELINE_14FOLD_STD:.4f}')
print(f'Delta (12-fold - 14-fold): mean {mean_acc - BASELINE_14FOLD_MEAN:+.4f}, '
      f'std {std_acc - BASELINE_14FOLD_STD:+.4f}')

summary = pd.DataFrame([
    dict(variant='14-fold (all subjects, baseline)', n_folds=14, accuracy_mean=BASELINE_14FOLD_MEAN, accuracy_std=BASELINE_14FOLD_STD),
    dict(variant='12-fold (excl. subject_22, subject_26)', n_folds=12, accuracy_mean=mean_acc, accuracy_std=std_acc),
])
summary.to_csv(OUT_DIR / 'subject_exclusion_robustness.csv', index=False)
summary

Loaded precomputed per-fold results (RUN_TRAINING=False); see thesis_analysis/03_subject_exclusion_robustness.py for the standalone run.

12-fold (excl. subject_22, subject_26) EEGNet LOSO: mean=0.6200 std=0.0586
14-fold baseline (all subjects): mean=0.6449 std=0.0573
Delta (12-fold - 14-fold): mean -0.0249, std +0.0013


,variant,n_folds,accuracy_mean,accuracy_std
0,"14-fold (all subjects, baseline)",14,0.644900,0.057300
1,"12-fold (excl. subject_22, subject_26)",12,0.620009,0.058562
